In [0]:
from pyspark.sql.functions import *

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.gold;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.gold.dim_geografia (
    id_ubigeo STRING COMMENT 'Código único de distrito según INEI',
    id_ubigeo_reniec STRING COMMENT 'Código único de distrito según RENIEC',
    nom_departamento STRING COMMENT 'Nombre del departamento',
    nom_provincia STRING COMMENT 'Nombre de la provincia',
    nom_distrito STRING COMMENT 'Nombre del distrito',
    latitud DOUBLE COMMENT 'Latitud geográfica del distrito',
    longitud DOUBLE COMMENT 'Longitud geográfica del distrito',
    fecha_data DATE COMMENT 'Fecha de actualización de la tabla Gold'
) USING DELTA;

In [0]:
df_ubi_equi = spark.table("workspace.silver.m_ubigeo_equivalencia").alias("A")
df_ubi_geo = spark.table("workspace.silver.m_ubigeo_geografia").alias("B")

dim_geografia = (df_ubi_equi.join(df_ubi_geo, "id_ubigeo_reniec", "left").select(
        "A.id_ubigeo",
        "B.id_ubigeo_reniec",
        "B.nom_departamento",
        "B.nom_provincia",
        "B.nom_distrito",
        "B.latitud",
        "B.longitud"
    )
    .withColumn("fecha_data", current_date())
)

dim_geografia.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.dim_geografia")

In [0]:
%sql
select * from workspace.gold.dim_geografia